In [ ]:
# Lets import the needed libraries. There are some quick annotations for what the unusual ones are used for.

import requests
import time
import pandas as pd
import os
from datetime import datetime
import yfinance as yf # Gets financial data.
from bs4 import BeautifulSoup # Parses HTML (for data cleaning).
from sentence_transformers import SentenceTransformer, util # Used to filter data for relevancy.
from dateutil.relativedelta import relativedelta  # Similar to normal datetime, but is calander aware, allowing us to +- months.
from newsapi import NewsApiClient # One of the API's used to get news data.
from dotenv import load_dotenv # Adding secret API keys.

# Below we load our API keys from the .env. You will require your own API keys, this is for the purpose of hiding mine.
# You will require a Guardian News API key, AlphaVantage API key, and an Newsapi API key, links and info in README.

load_dotenv()
guardian_api_token = os.getenv('guardian_TOKEN') 
newsapi_token = os.getenv('newsapi_TOKEN')
Alphavantage_token = os.getenv('ALPHA_TOKEN')
newsapi = NewsApiClient(api_key=newsapi_token)

In [ ]:
# This function gets news data (we eventually take date, headline and body text after cleaning).
# It gets any data (that we search for) from the Guardian News.

def get_content_from_guardian(searches,start_date,end_date,api_key):
    content = []
    # Is it 50 articels per page, so we go through multiple pages that match with the same date needed.
    for i in searches:
        time.sleep(5)  # Worked before, but sometimes got connection blocked, this seems to have fixed it. You may remove it and it may work, API seems rather temprimental.
        page = 1                             
        while True: # We keep going through while there are more searches and pages.
            query = requests.get('https://content.guardianapis.com/search',params={'q' :i,'from-date':start_date,'to-date': end_date,'api-key': api_key,'page-size'  : 50,'page'       : page,
                                                                                    'show-fields': 'headline,body'})
            data = query.json()['response']
            content.extend(data['results'])
            if page >= data['pages']: # Moving onto the next page.
                break
            page += 1
    return content

searches = [ # All the searches we want to input.
    '"Iran war" AND oil',
    '"Iran war" AND energy',
    '"Iran war" AND markets',
    '"Iran war" AND prices',
    '"oil price" AND Iran',
    'Strait of Hormuz',
    'Iran oil June 2025',
    'crude oil',
    '"crude oil" AND Iran AND war',
    '"energy market" AND Iran AND war',
    'energy market',
    'commodity market AND war',
    '"oil supply" AND Iran AND war',
    '"investor" AND Iran AND oil',
    '"stock market" AND energy',
    'stocks',
    'stock market',
]

# Calling the above function and storing as a pandas dataframe.
raw_guardian_df = pd.DataFrame(get_content_from_guardian(searches, '2026-02-01',str(datetime.today().date()),guardian_api_token))

In [ ]:
# Creating a copy of the dataframe so if there are issues in guardian_df we dont have to re run the above function.

guardian_df = raw_guardian_df.copy()

In [ ]:
# Some initial data cleaning.

guardian_df['date'] = pd.to_datetime(guardian_df['webPublicationDate']).dt.strftime('%Y%m%d') # Formatting date to year-month-day.
guardian_df.drop_duplicates(subset=['webUrl'], inplace=True) # Dropping any duplicates as they may have been picked up multiple times from the different searches.
guardian_df = guardian_df[['date','fields']] # We only need date and fields, fields is a dictionary of headline and body text.
guardian_df.reset_index(drop=True, inplace=True)

In [ ]:
# This cell extracts the headline and body text from the fields dictionary.

guardian_df['headline'] = None
guardian_df['body_text'] = None

for i, item in enumerate(guardian_df['fields']):
    guardian_df.at[i, 'headline'] = item['headline'] # Headline is easy to get.
    soup = BeautifulSoup(item['body']) # Body text contains alot of HTML, so we clean it with beautiful soup.
    body_text = soup.get_text()
    guardian_df.at[i, 'body_text'] = body_text

In [ ]:
# 25req/day, 5req/min
# this should only use 4
# Used for more recent data as other APIs have larger delays in upload/article add to index times.

# We use a similar structure for the get_content_from_guardian function.

def get_content_from_alphavantage(searches,start_date,end_date):
    content = []
    for i in searches:
        time.sleep(12)
        query = requests.get('https://www.alphavantage.co/query', params= {'function':'NEWS_SENTIMENT','topics':i,'time_from':start_date,'time_to':end_date,'apikey':Alphavantage_token
                                                                           ,'limit':1000,'sort':'LATEST'})
        data = query.json()
        content.extend(data.get('feed',[]))
    return content

# Have to choose from premade topics on the api doc page (link in README.)
searches = ['energy_transportation','financial_markets','economy_macro','finance']

# Calling and saving function to dataframe, this time it requires a different date format.
raw_alphavantage_df = pd.DataFrame(get_content_from_alphavantage(searches,'20260201T0000',datetime.today().strftime('%Y%m%dT%H%M')))

In [ ]:
# Cleaning the alphavantage data, converting date to the same as gaurdian. We only take the needed columns.

alphavantage_df = raw_alphavantage_df[['title','summary','time_published']]
alphavantage_df['date'] = pd.to_datetime(alphavantage_df['time_published']).dt.strftime('%Y%m%d')
alphavantage_df = alphavantage_df[['title','summary','date']]
alphavantage_df.reset_index(drop=True, inplace=True)

In [ ]:
# We are now moving on to the NewsAPI section. On its free tier, this API can only go back one month. It has been run daily to build up a database that goes back further -
# - than one month. We do this by generating a pair of dates (required as inputs) and use them to re run the API daily with new date inputs.
# These dates are todays current date, and the date one month ago.
# This function returns that date pair.

def get_date_pairs():
    current_date = datetime.today().date()
    start_date = current_date - relativedelta(months=1)

    date_pairs = []

    while start_date<current_date:
        next_day_date = start_date + relativedelta(days=1)
        date_pairs.append((start_date,next_day_date))
        start_date = next_day_date

    return date_pairs

In [ ]:
# Here is the actual NewsAPI function, where we can only search back for one month.
# By running daily, using the generated date pairs each day, and storing it, we have built up a larger database that goes back longer than the 1 month limit allows.

def get_newsAPI_data():
    news_data = []
    for i in get_date_pairs():
        all_top_headlines = newsapi.get_everything(q='(oil OR crude OR energy OR brent OR wti) AND (iran OR price OR prices OR supply OR Stocks OR Energy market or stock market or commodity market OR demand OR production OR disruption OR volatility OR surge OR drop)',
                                           from_param=i[0], # Inputting dates.
                                           to=i[1], # Inputting dates.
                                           language='en', # Only english language.
                                           sort_by='relevancy') # Sorting by relevancy.
        articles = all_top_headlines['articles']
        news_data.extend(articles)
    return news_data
    
news_data = get_newsAPI_data()

In [ ]:
# Data cleaning and initial 'saving' of NewsAPI data

normalised_dataframe = pd.DataFrame(pd.json_normalize(news_data)) # Converts nested data into single columns
normalised_dataframe.drop_duplicates(subset=['title'],inplace=True) # Dropping any duplicates

# As this NewsAPI is designed to be run daily to build up a database, if the csv file already exists, we just add onto it.
# However if this is the first time it is run, we create a new file.

if os.path.exists('../Data/Data_Pre_Sentiment/Working_data_for_newsAPI(not_output)/NewsAPI_data_multipleday.csv'): 
    normalised_dataframe.to_csv('../Data/Data_Pre_Sentiment/Working_data_for_newsAPI(not_output)/NewsAPI_data_multipleday.csv', index=False, mode='a', header=False, encoding='utf-8') # Had some encoding issues, now we just ensure its utf-8
    print('Dataset updated')

else:
    normalised_dataframe.to_csv('../Data/Data_Pre_Sentiment/Working_data_for_newsAPI(not_output)/NewsAPI_data_multipleday.csv', index=False, encoding='utf-8')
    print('File does not exist, creating new')

In [ ]:
# Now that we have updated, or created, the NewsAPI file, we can call the entire file, complete with data going back further than the one month limit.

NewsAPI_to_get_relevant = pd.read_csv('../Data/Data_Pre_Sentiment/Working_data_for_newsAPI(not_output)/NewsAPI_data_multipleday.csv',encoding='utf-8')
NewsAPI_to_get_relevant.drop_duplicates(subset='title', inplace=True) # Incase any duplicates got past.
NewsAPI_to_get_relevant['date'] = pd.to_datetime(NewsAPI_to_get_relevant['publishedAt']).dt.strftime('%Y%m%d') # Ensuring date is the correct format.


In [ ]:
# While we only download the data we need, some results are still not relevant.
# This model checks that each headline is relevant, if it is not, it is not added to the final news dataframe (combining NewsAPI and Guardian)

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2') # We use the all-MiniLM-L6-v2 model from hugging face.
embeddings = model.encode('oil price energy markets Iran war investor sentiment volatility crude supply',convert_to_tensor=True) # This is what we compare relevancy to.

def relevancy(headline): # Creating a function to compare the headline to the relevancy baseline.
    temp = model.encode(headline, convert_to_tensor=True)
    return util.cos_sim(temp,embeddings).item()

# We only take data with a relevancy of 0.35 or higher for guardian and newsapi. alphavantage is already very specific so 0.3.

guardian_df['relevancy'] = guardian_df['headline'].apply(relevancy)
guardian_df = guardian_df[guardian_df['relevancy'] > 0.35]

NewsAPI_to_get_relevant['relevancy'] = NewsAPI_to_get_relevant['title'].apply(relevancy)
NewsAPI_to_get_relevant = NewsAPI_to_get_relevant[NewsAPI_to_get_relevant['relevancy'] > 0.35]

alphavantage_df['relevancy'] = alphavantage_df['title'].apply(relevancy)
alphavantage_df = alphavantage_df[alphavantage_df['relevancy'] > 0.30]


In [ ]:
# We now have all the news data needed, we move onto getting financial data.
# We use yfinance to download data on Brent, WTI, OVX (Oil volatility index), VIX (market volatility index), and from the S&P 500 index.

# Creating a function we can reuse to download the needed data and filter it.

def get_finance_data(ticker):
    pre_data = yf.download(ticker, start='2026-02-01', end=datetime.today().date()) # Downloading data.
    pre_data.reset_index(inplace=True) # Resetting its index.
    pre_data['daily_avg'] = pre_data[['Open','Close']].mean(axis=1) # For the value (per day) we make it as an average of the days opening and closing price.
    data = pre_data[['Date','daily_avg','Close']]
    data.columns = ['date','daily_avg','Close'] # Helps format for later.
    data['date'] = pd.to_datetime(data['date'], format='%Y%m%d') # Ensuring date is the correct format.
    
    # As some data wont be available on certain days (e.g Brent wont have data on non trading days such as the weekend) - 
    # - we fill that empty value with the previous days closing price.

    time = pd.DataFrame({'date': pd.date_range(start='2026-02-01',end=datetime.today().date(),freq='D')}) # We create a dataframe of all the dates used, including non trading days.

    data = time.merge(data, on='date', how='left') # We add these non trading day dates into the main dataframe.
    data['Close'] = data['Close'].ffill() # We use a forward fill to take the previous days closing price forward if the next value is a null.
    data['daily_avg'] = data['daily_avg'].fillna(data['Close']) # We ensure they are all stored under the same column.
    data = data[['date','daily_avg']] # We only need the data and its value.
    return data

# Below we call the function for each ticker we want (listed at top of cell), and save it to a csv.

get_finance_data('BZ=F').to_csv('../Data/Data_Pre_Sentiment/Brent_data.csv',index=False, encoding='utf-8')
get_finance_data('CL=F').to_csv('../Data/Data_Pre_Sentiment/WTI_data.csv',index=False, encoding='utf-8')
get_finance_data('^OVX').to_csv('../Data/Data_Pre_Sentiment/ovx_data.csv',index=False, encoding='utf-8')
get_finance_data('^VIX').to_csv('../Data/Data_Pre_Sentiment/vix_data.csv',index=False, encoding='utf-8')
get_finance_data('^SPX').to_csv('../Data/Data_Pre_Sentiment/S&P500_data.csv',index=False, encoding='utf-8')


In [ ]:
# In this cell we are saving data we scraped from the Guardian Newspaper, alphavantage, and from NewsAPI.

guardian_df = guardian_df[['date','headline','body_text']]
guardian_df.reset_index(drop=True, inplace=True)
guardian_df.to_csv('../Data/Data_Pre_Sentiment/guardian_data.csv', index=False, encoding='utf-8')

NewsAPI_to_get_relevant = NewsAPI_to_get_relevant[['date','title','content']]
NewsAPI_to_get_relevant.reset_index(drop=True, inplace=True)
NewsAPI_to_get_relevant.to_csv('../Data/Data_Pre_Sentiment/NewsAPI_data.csv', index=False, encoding='utf-8')

alphavantage_df = alphavantage_df[['date','title','summary']]
alphavantage_df.reset_index(drop=True, inplace=True)
alphavantage_df.to_csv('../Data/Data_Pre_Sentiment/alphavantage_data.csv', index=False, encoding='utf-8')

